In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

/home/aiml/Documents/Projects/AgenticAi/.venv/lib64/python3.14/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [6]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [7]:
# execute

initial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])



The Moon orbits the Earth at an average distance of **about 384 400 kilometers (≈ 238 900 miles)**.  

- **Perigee (closest point)**: ~ 356 500 km (≈ 221 500 mi)  
- **Apogee (farthest point)**: ~ 406 700 km (≈ 252 700 mi)  

Because the Moon’s orbit is slightly elliptical, the exact distance varies within this range, but the 384 400 km figure is the commonly quoted average.


In [9]:
model.invoke('How far is moon from the earth?').content

'The Moon orbits Earth at an average distance of **about\u202f384\u202f400\u202fkilometers (≈\u202f238\u202f900\u202fmiles)**.\n\nBecause the Moon’s orbit is slightly elliptical, the actual distance changes over the course of each month:\n\n| Orbital point | Approximate distance from Earth |\n|---------------|---------------------------------|\n| **Perigee** (closest) | ~363\u202f300\u202fkm (≈\u202f225\u202f600\u202fmi) |\n| **Apogee** (farthest) | ~405\u202f500\u202fkm (≈\u202f251\u202f900\u202fmi) |\n| **Mean (average)**   | ~384\u202f400\u202fkm (≈\u202f238\u202f900\u202fmi) |\n\nThese numbers are based on the **geocentric (Earth‑centered) distance** to the Moon’s center. If you measure from the surface of Earth to the surface of the Moon, subtract roughly 6\u202f370\u202fkm (Earth’s radius) and 1\u202f740\u202fkm (Moon’s radius), which yields a surface‑to‑surface range of about **376\u202f000\u202fkm to 398\u202f000\u202fkm**.'